Report of several practical examples where people have automated downloads (or file retrieval) with Python. Many examples you see on Reddit, developer blogs, and tutorials show that—whether it’s downloading a report, a media file, or scraping content for later use—Python’s rich ecosystem makes it straightforward. Here are eight representative examples with concise code snippets:

---

### 1. Basic File Download with `urllib`

A common starting point is to use Python’s built‐in `urllib.request.urlretrieve` to download a file from an HTTP URL. For example:

In [ ]:
import urllib.request

url = "https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf"
filepath = "../../../assets/paper3.pdf"
urllib.request.urlretrieve(url, filepath)
print("Downloaded paper")

In [ ]:
!ls ../../../assets/*.pdf* | grep paper

This can be handy when you want a one-liner that mimics the familiar Unix tool.  


---

### 2. Conditional Download (Based on File Age)

Sometimes you want to download a file only if it’s missing or older than a given age. For example, to download a CSV file if it’s older than one day:

In [ ]:
import urllib.request
import os
import time

url = 'https://gist.githubusercontent.com/denandreychuk/b9aa812f10e4b60368cff69c6384a210/raw/100%20Sales%20Records.csv'
local_file = 'data.csv'
# Download if file doesn't exist or is older than 24 hours (86400 seconds)
if not os.path.exists(local_file) or (os.path.getmtime(local_file) < time.time() - 86400):
    urllib.request.urlretrieve(url, local_file)
    print("Updated data.csv")
else:
    print("data.csv is up-to-date")

In [ ]:
import os
import time
import requests

# fake csv file (doesn't exist online!)
url = 'https://gist.githubusercontent.com/denandreychuk/b9aa812f10e4b60368cff69c6384a210/raw/100%20Sales%20Records.csv'
local_file = 'data.csv'

def download_if_needed(url, local_file):
    # 24 hours in seconds
    max_age = 24 * 60 * 60  

    # Condition 1 + 2: Check if file exists AND whether it is older than 24h
    needs_download = (
        not os.path.exists(local_file) or
        (time.time() - os.path.getmtime(local_file)) > max_age
    )

    if needs_download:
        print("Downloading fresh copy...")
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            raise RuntimeError("Download failed.")
        with open(local_file, "wb") as f:
            f.write(resp.content)
    else:
        print("Local file is recent; no download needed.")

# Run it
download_if_needed(url, local_file)

In [ ]:
!ls *.csv

This pattern is popular in automating daily report updates.  


---

### 3. Downloading Large Files with `requests` and Streaming

When downloading large files, it’s best to stream the response in chunks. This approach uses the `requests` library:

In [ ]:
import requests

url = "http://example.com/largefile.zip"
local_filename = "largefile.zip"
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(local_filename, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
print("Large file downloaded successfully.")

Using streaming helps avoid high memory usage with large downloads.  


---

### 4. Automating Downloads via Selenium

For sites that require a login or button clicks (for instance, downloading a report from a secure portal), Selenium can be used to simulate user actions:

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

driver = webdriver.Chrome()  # Assumes chromedriver is installed and in PATH
driver.get("http://example.com/login")
# Fill in login form
driver.find_element(By.ID, "username").send_keys("myusername")
driver.find_element(By.ID, "password").send_keys("mypassword" + Keys.RETURN)
time.sleep(3)  # Wait for login to complete
# Navigate to download page and click the download button
driver.get("http://example.com/download")
driver.find_element(By.ID, "downloadButton").click()
time.sleep(5)  # Wait for download to start/complete
driver.quit()

This method is frequently mentioned in discussions where users automate downloading reports or files from platforms that don’t offer direct URL-based downloads.  


---

### 5. Downloading Videos Using `youtube_dl`

For media files (especially videos from platforms like YouTube), many developers rely on the popular tool `youtube_dl` (which now has forks such as `yt-dlp`):

In [ ]:
import youtube_dl

ydl_opts = {}
with youtube_dl.YoutubeDL(ydl_opts) as ydl:
    ydl.download(['https://www.youtube.com/watch?v=EXAMPLE_ID'])

This snippet automatically downloads the video file using the robust features of the youtube_dl ecosystem.  


---

### 6. Invoking the Command-line `wget` from Python

Sometimes you want to leverage the battle-tested command-line utility wget (especially for features like resuming downloads). You can invoke it via Python’s subprocess module:

In [ ]:
import subprocess

url = "http://example.com/file.zip"
# The "-c" flag allows resuming an interrupted download.
subprocess.run(["wget", "-c", url])

This is useful when you’re comfortable with wget’s features and want to combine them with your Python workflow.  


---

### 7. Concurrent Downloads Using `aiohttp` and `asyncio`

For scenarios where you need to download many files at once, asynchronous code with `aiohttp` can dramatically speed things up:

In [ ]:
import aiohttp
import asyncio

async def download_file(session, url, filename):
    async with session.get(url) as resp:
        with open(filename, 'wb') as f:
            while True:
                chunk = await resp.content.read(1024)
                if not chunk:
                    break
                f.write(chunk)
    print(f"Downloaded {filename}")

async def main():
    files = [
        ("http://example.com/file1.zip", "file1.zip"),
        ("http://example.com/file2.zip", "file2.zip"),
        # Add more (url, filename) tuples as needed.
    ]
    async with aiohttp.ClientSession() as session:
        tasks = [download_file(session, url, fname) for url, fname in files]
        await asyncio.gather(*tasks)

asyncio.run(main())

This approach uses asynchronous programming to start multiple downloads concurrently, which is ideal when processing many files in bulk.  

# Example for Organizing Downloads Folder

# Exercise for writing prompts to ChatGPT

## Create a prompt for ChatGPT to create a script to perform something like organizing files in a folder, but adding your own twist, perspective or usecase.

**Prompt to ChatGPT (GPT-4o)**
```
Write a Python script that organizes files in my 'Downloads' folder by moving them into subfolders based on their file type (e.g., PDFs to a 'PDFs' folder, images to an 'Images' folder, etc.). The script should create folders if they don’t exist.
```

In [ ]:
import os
import shutil
from collections import defaultdict

# ⚠️ WARNING: this points at your REAL Downloads folder and will actually move
# files there into type-named subfolders when you run organize_downloads() below.
# This cell is a template to read, not something to run as part of following
# along with the notebook — only run it if you're OK reorganizing ~/Downloads.
DOWNLOADS_DIR = os.path.expanduser("~/Downloads")

# Define file type categories
FILE_CATEGORIES = {
    "Images": [".jpg", ".jpeg", ".png", ".gif", ".bmp", ".svg", ".tiff"],
    "Documents": [".pdf", ".doc", ".docx", ".txt", ".odt", ".rtf", ".tex"],
    "Spreadsheets": [".xls", ".xlsx", ".csv", ".ods"],
    "Presentations": [".ppt", ".pptx", ".key"],
    "Archives": [".zip", ".rar", ".tar", ".gz", ".7z"],
    "Executables": [".exe", ".dmg", ".pkg", ".app"],
    "Videos": [".mp4", ".mkv", ".mov", ".avi", ".flv", ".wmv"],
    "Music": [".mp3", ".wav", ".flac", ".aac", ".ogg", ".m4a"],
    "Code": [".py", ".js", ".html", ".css", ".java", ".c", ".cpp", ".go", ".sh", ".ipynb"]
}

# Reverse the dictionary to map file extensions to their respective categories
EXTENSION_TO_CATEGORY = {ext: category for category, exts in FILE_CATEGORIES.items() for ext in exts}

# Ensure categorized folders exist
def create_folders():
    for folder in FILE_CATEGORIES.keys():
        folder_path = os.path.join(DOWNLOADS_DIR, folder)
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)

# Organize files
def organize_downloads():
    create_folders()
    
    for filename in os.listdir(DOWNLOADS_DIR):
        file_path = os.path.join(DOWNLOADS_DIR, filename)
        
        # Skip directories
        if os.path.isdir(file_path):
            continue
        
        # Get the file extension
        _, ext = os.path.splitext(filename)
        ext = ext.lower()
        
        # Determine the category
        category = EXTENSION_TO_CATEGORY.get(ext, "Other")
        
        # Create 'Other' folder if needed
        target_folder = os.path.join(DOWNLOADS_DIR, category)
        if not os.path.exists(target_folder):
            os.makedirs(target_folder)
        
        # Move the file
        shutil.move(file_path, os.path.join(target_folder, filename))
        print(f"Moved: {filename} -> {category}/")

# Not called automatically — uncomment only if you want to actually
# reorganize your real ~/Downloads folder.
# if __name__ == "__main__":
#     organize_downloads()

---

## Part 2 — Data Analysis with AI

The cells below were merged in from the former `03-data-analysis.ipynb`. They walk through the AI-assisted data-analysis workflow: showing a data sample to an AI model, asking for analysis code, and inspecting the results.


1. Showing a sample of the data to an AI model like ChatGPT
2. Asking for the Python code for the analysis
3. Inspecting and running the code
4. Automating it locally in our machines to run forever

In [ ]:
import pandas as pd

df = pd.read_csv("../../../assets/others/stock-trading-data.csv")

df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as sns
import os

# Load data
def load_data(file_path):
    df = pd.read_csv(file_path, parse_dates=['Date'])
    df.dropna(inplace=True)  # Handle missing values
    df.sort_values(by=['Date'], inplace=True)
    return df

# Compute daily returns and rolling volatility
def compute_returns_volatility(df):
    df['Daily Return'] = df.groupby('Stock Symbol')['Close Price'].pct_change()
    df['Rolling Volatility'] = df.groupby('Stock Symbol')['Daily Return'].rolling(window=20).std().reset_index(level=0, drop=True)
    return df

# Compute SMA and EMA
def compute_moving_averages(df):
    df['SMA_20'] = df.groupby('Stock Symbol')['Close Price'].transform(lambda x: x.rolling(window=20).mean())
    df['EMA_20'] = df.groupby('Stock Symbol')['Close Price'].transform(lambda x: x.ewm(span=20, adjust=False).mean())
    return df

# Compute Bollinger Bands
def compute_bollinger_bands(df):
    rolling_mean = df.groupby('Stock Symbol')['Close Price'].transform(lambda x: x.rolling(window=20).mean())
    rolling_std = df.groupby('Stock Symbol')['Close Price'].transform(lambda x: x.rolling(window=20).std())
    df['Upper Band'] = rolling_mean + (rolling_std * 2)
    df['Lower Band'] = rolling_mean - (rolling_std * 2)
    return df

# Compute MACD
def compute_macd(df):
    df['EMA_12'] = df.groupby('Stock Symbol')['Close Price'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
    df['EMA_26'] = df.groupby('Stock Symbol')['Close Price'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['Signal Line'] = df.groupby('Stock Symbol')['MACD'].transform(lambda x: x.ewm(span=9, adjust=False).mean())
    return df

# Generate visualizations
def plot_stock_prices(df, symbol):
    stock_df = df[df['Stock Symbol'] == symbol]
    plt.figure(figsize=(12, 6))
    plt.plot(stock_df['Date'], stock_df['Close Price'], label='Close Price')
    plt.plot(stock_df['Date'], stock_df['SMA_20'], label='20-day SMA', linestyle='dashed')
    plt.plot(stock_df['Date'], stock_df['EMA_20'], label='20-day EMA', linestyle='dotted')
    plt.fill_between(stock_df['Date'], stock_df['Upper Band'], stock_df['Lower Band'], color='gray', alpha=0.3, label='Bollinger Bands')
    plt.title(f'{symbol} Stock Price Analysis')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.legend()
    plt.grid()
    plt.show()

# Generate heatmap of correlations
def plot_correlation_heatmap(df):
    numeric_cols = ['Close Price', 'Daily Return', 'Rolling Volatility', 'SMA_20', 'EMA_20', 'RSI (Relative Strength Index)', 'MACD', 'Signal Line']
    correlation_matrix = df[numeric_cols].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Stock Metric Correlation Heatmap')
    plt.savefig('correlation_heatmap.png')
    plt.close()

# Export analysis to Excel
def export_to_excel(df):
    output_file = 'financial_analysis_report.xlsx'
    with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
        df.to_excel(writer, sheet_name='Stock Data', index=False)
        workbook = writer.book
        worksheet = writer.sheets['Stock Data']
        worksheet.insert_image('J2', 'AAPL_stock_price.png')
        worksheet.insert_image('J20', 'correlation_heatmap.png')
    print(f'Report saved as {output_file}')

# Main function



df = compute_returns_volatility(df)
df = compute_moving_averages(df)
df = compute_bollinger_bands(df)
df = compute_macd(df)

plot_stock_prices(df, 'AAPL')
# plot_correlation_heatmap(df)
# export_to_excel(df)